# Methods Extension — Progression‑Sensitive Biomarker Pipeline (FRDA)

**Goal:** learn progression‑sensitive biomarkers that separate **Visit 2 vs Visit 1** within subject.

Primary metric: **Cohen’s d** (paired V2−V1 on out‑of‑fold scores).
Complementary metric: **SRM** (reported alongside d).

Secondary metrics (regression tasks): **RMSE** and **R²** for predicting **FARS1** and **ΔFARS**.

**Entropy / information integration:**
1) Individual‑feature mutual information (MI) analysis (visit, FARS1, ΔFARS)
2) Entropy‑guided feature selection (top‑k per group and top‑k global) evaluated with the same CV protocol.

Outputs are saved under `baseline_composite_biomakers/notebooks/`.


In [1]:
# SECTION A — Imports, config, paths
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Note: This notebook intentionally avoids hard dependency on scikit-learn.
# If scikit-learn is available in your environment, you can swap in sklearn models easily.

RANDOM_SEED = 42
rng_global = np.random.RandomState(RANDOM_SEED)

BUNDLE_ROOT = Path('..').resolve()
os.chdir(BUNDLE_ROOT)
print('Working directory:', Path.cwd())

DATA_PATH = Path('data/processed/melbourne_frda_merged.csv')
OUTPUT_DIR = Path('results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

K_VALUES = [1, 2, 3, 5]
N_BOOT = 2000

pd.set_option('display.max_columns', 200)


Working directory: /Users/robertwang/Documents/New_project/baseline_composite_biomakers


## SECTION A — Feature registry

Feature groups (as discussed with supervisor) are defined explicitly:
- `structural` baseline set
- `structural_ext` = structural + CSA/ECC extensions (treated as an extension, not a replacement)


In [2]:
# Feature groups
background = [
    'age', 'age_at_onset', 'dur', 'sex',
    'GAA1', 'GAA2'
]

structural = [
    'SCP', 'MCP', 'ICP',
    'Midbrain', 'Pons', 'Medulla',
    'AntCBLM', 'SupPostCBLM', 'InfPostCBLM', 'FlocCBLM', 'VermisCBLM'
]

structural_ext = structural + ['CSA_C1', 'CSA_C2', 'ECC_C1', 'ECC_C2']

diffusion = [
    'FASCP', 'FAMCP', 'FAICP',
    'MDSCP', 'MDMCP', 'MDICP',
    'ADSCP', 'ADMCP', 'ADICP',
    'RDSCP', 'RDMCP', 'RDICP'
]

# Explicit flat feature sets
FEATURE_SETS = {
    'background': background,
    'structural': structural,
    'structural_ext': structural_ext,
    'diffusion': diffusion,
    'background_structural': background + structural,
    'background_structural_ext': background + structural_ext,
    'background_diffusion': background + diffusion,
    'structural_diffusion': structural + diffusion,
    'structural_ext_diffusion': structural_ext + diffusion,
    'background_structural_diffusion': background + structural + diffusion,
    'background_structural_ext_diffusion': background + structural_ext + diffusion,
}

# Global pool used for entropy-selected subsets
ALL_FEATURES = sorted(set(background + structural_ext + diffusion))

FEATURE_GROUPS = {
    'background': background,
    'structural': structural,
    'structural_ext': structural_ext,
    'diffusion': diffusion,
}

print('Feature sets:', len(FEATURE_SETS))
print('Global feature pool:', len(ALL_FEATURES))


Feature sets: 11
Global feature pool: 33


## SECTION B — Data loading, validation, and dataset views

We build three aligned views:
- `df_long`: 2 rows/subject (visit 1/2) for separation models (e.g. LDA)
- `df_baseline`: 1 row/subject at V1 for regression target **FARS1**
- `df_delta`: 1 row/subject of changes (V2−V1) for regression target **ΔFARS**

All evaluation is subject-level (no leakage across visits).


In [3]:
# Load wide data and detect subject column
wide = pd.read_csv(DATA_PATH)

for _c in ['melb_id', 'ID', 'subject', 'subject_id']:
    if _c in wide.columns:
        subject_col = _c
        break
else:
    raise KeyError('No subject id column found')

print('Wide shape:', wide.shape)
print('Subject column:', subject_col, '| n_subjects:', wide[subject_col].nunique())

# Targets required by this notebook
required_targets = ['FARS1', 'FARS2']
missing = [c for c in required_targets if c not in wide.columns]
if missing:
    raise KeyError(f'Missing required target columns: {missing}')

# Helper: confirm visit-specific feature columns exist
v1_cols = [c for c in wide.columns if c.endswith('_v1')]
v2_cols = [c for c in wide.columns if c.endswith('_v2')]
base_cols = sorted({c[:-3] for c in v1_cols} & {c[:-3] for c in v2_cols})

# Keep only bases we recognize (structural_ext + diffusion); background handled separately
imaging_bases = [b for b in base_cols if b in set(structural_ext + diffusion)]

print('Imaging bases found:', len(imaging_bases))

# Build df_long
rows = []
for _, r in wide.iterrows():
    sid = r[subject_col]
    # visit 1
    row1 = {subject_col: sid, 'visit': 1}
    row1['age'] = r.get('age1')
    row1['dur'] = r.get('dur1')
    row1['age_at_onset'] = r.get('age_at_onset')
    row1['sex'] = r.get('sex')
    row1['GAA1'] = r.get('GAA1')
    row1['GAA2'] = r.get('GAA2')
    row1['FARS'] = r.get('FARS1')
    if 'SARA1' in wide.columns:
        row1['SARA'] = r.get('SARA1')

    for f in imaging_bases:
        row1[f] = r.get(f + '_v1')

    # visit 2
    row2 = {subject_col: sid, 'visit': 2}
    row2['age'] = r.get('age2')
    row2['dur'] = r.get('dur2')
    row2['age_at_onset'] = r.get('age_at_onset')
    row2['sex'] = r.get('sex')
    row1['GAA1'] = r.get('GAA1')
    row2['GAA2'] = r.get('GAA2')
    row2['FARS'] = r.get('FARS2')
    if 'SARA2' in wide.columns:
        row2['SARA'] = r.get('SARA2')

    for f in imaging_bases:
        row2[f] = r.get(f + '_v2')

    rows.append(row1)
    rows.append(row2)

df_long = pd.DataFrame(rows)

# Basic validation
assert set(df_long['visit'].unique()).issubset({1, 2})

# Ensure complete pairs exist
pair_counts = df_long.groupby(subject_col)['visit'].nunique()
paired_subjects = pair_counts[pair_counts == 2].index
print('Long shape:', df_long.shape)
print('Paired subjects:', len(paired_subjects), '/', wide[subject_col].nunique())

# Build df_baseline (V1 row per subject)
# For delta tasks, we keep the same feature names, but values are changes (V2-V1) where appropriate.
baseline_rows = []
delta_rows = []

for _, r in wide.iterrows():
    sid = r[subject_col]

    # Baseline
    br = {subject_col: sid}
    br['FARS1'] = r.get('FARS1')
    if 'SARA1' in wide.columns:
        br['SARA1'] = r.get('SARA1')

    # background mapped to V1/static
    br['age'] = r.get('age1')
    br['dur'] = r.get('dur1')
    br['age_at_onset'] = r.get('age_at_onset')
    br['sex'] = r.get('sex')
    br['GAA1'] = r.get('GAA1')
    br['GAA2'] = r.get('GAA2')

    for f in imaging_bases:
        br[f] = r.get(f + '_v1')

    baseline_rows.append(br)

    # Delta
    dr = {subject_col: sid}
    f1 = r.get('FARS1')
    f2 = r.get('FARS2')
    dr['dFARS'] = np.nan if (pd.isna(f1) or pd.isna(f2)) else (f2 - f1)

    # background deltas + static
    a1, a2 = r.get('age1'), r.get('age2')
    d1, d2 = r.get('dur1'), r.get('dur2')
    dr['age'] = np.nan if (pd.isna(a1) or pd.isna(a2)) else (a2 - a1)
    dr['dur'] = np.nan if (pd.isna(d1) or pd.isna(d2)) else (d2 - d1)
    dr['age_at_onset'] = r.get('age_at_onset')
    dr['sex'] = r.get('sex')
    dr['GAA1'] = r.get('GAA1')
    dr['GAA2'] = r.get('GAA2')

    for f in imaging_bases:
        v1 = r.get(f + '_v1')
        v2 = r.get(f + '_v2')
        dr[f] = np.nan if (pd.isna(v1) or pd.isna(v2)) else (v2 - v1)

    delta_rows.append(dr)

df_baseline = pd.DataFrame(baseline_rows)
df_delta = pd.DataFrame(delta_rows)

print('Baseline df:', df_baseline.shape, '| non-missing FARS1:', df_baseline['FARS1'].notna().sum())
print('Delta df:', df_delta.shape, '| non-missing dFARS:', df_delta['dFARS'].notna().sum())


Wide shape: (26, 69)
Subject column: ID | n_subjects: 26
Imaging bases found: 27
Long shape: (52, 37)
Paired subjects: 26 / 26
Baseline df: (26, 36) | non-missing FARS1: 26
Delta df: (26, 35) | non-missing dFARS: 26


## SECTION C — Shared helpers: metrics, MI (entropy), CV splitting, and models

This notebook provides minimal numpy/pandas implementations to remain runnable in lightweight environments.


In [4]:
import math

# ---------- Utilities ----------

def _as_float_array(x):
    return np.asarray(x, dtype=float)


def standardize_train_test(X_train, X_test):
    X_train = _as_float_array(X_train)
    X_test = _as_float_array(X_test)
    mu = np.nanmean(X_train, axis=0)
    sd = np.nanstd(X_train, axis=0, ddof=0)
    sd = np.where(sd == 0, 1.0, sd)
    return (X_train - mu) / sd, (X_test - mu) / sd, mu, sd


def group_kfold_indices(groups, n_splits=5, seed=RANDOM_SEED):
    groups = np.asarray(groups)
    uniq = np.unique(groups)
    if len(uniq) < 2:
        return
    rs = np.random.RandomState(seed)
    rs.shuffle(uniq)
    folds = np.array_split(uniq, min(n_splits, len(uniq)))
    for fold_groups in folds:
        val_mask = np.isin(groups, fold_groups)
        train_idx = np.where(~val_mask)[0]
        val_idx = np.where(val_mask)[0]
        if len(np.unique(groups[val_idx])) == 0 or len(np.unique(groups[train_idx])) == 0:
            continue
        yield train_idx, val_idx


def paired_deltas_from_long(oof_df, subject_col, visit_col, value_col):
    required = [subject_col, visit_col, value_col]
    if oof_df is None or any(c not in oof_df.columns for c in required):
        return pd.Series(dtype=float)
    tmp = oof_df[required].dropna().copy()
    if tmp.empty:
        return pd.Series(dtype=float)
    paired = (tmp.sort_values([subject_col, visit_col])
                .groupby(subject_col)[value_col]
                .apply(list))
    paired = paired[paired.map(len) == 2]
    if len(paired) == 0:
        return pd.Series(dtype=float)
    deltas = paired.map(lambda x: float(x[1] - x[0]))
    return deltas


def compute_cohens_d(deltas):
    deltas = _as_float_array(deltas)
    deltas = deltas[np.isfinite(deltas)]
    n = int(len(deltas))
    if n < 2:
        return {'d': np.nan, 'n': n, 'mean': np.nan, 'sd': np.nan}
    mean = float(np.mean(deltas))
    sd = float(np.std(deltas, ddof=1))
    d = np.nan if sd == 0 else float(mean / sd)
    return {'d': d, 'n': n, 'mean': mean, 'sd': sd}


def compute_srm(deltas):
    # SRM for paired change is numerically the same formula as paired Cohen's d.
    out = compute_cohens_d(deltas)
    return {'srm': out['d'], 'n': out['n'], 'mean': out['mean'], 'sd': out['sd']}


def bootstrap_ci_d(oof_df, subject_col, visit_col, value_col, n_boot=N_BOOT, seed=RANDOM_SEED):
    rs = np.random.RandomState(seed)
    deltas = paired_deltas_from_long(oof_df, subject_col, visit_col, value_col)
    if deltas.empty or deltas.shape[0] < 2:
        return (np.nan, np.nan, np.nan)
    subjects = deltas.index.to_numpy()
    vals = deltas.to_numpy(dtype=float)
    boot = []
    for _ in range(int(n_boot)):
        idx = rs.randint(0, len(subjects), size=len(subjects))
        sample = vals[idx]
        out = compute_cohens_d(sample)
        boot.append(out['d'])
    boot = np.asarray(boot, dtype=float)
    boot = boot[np.isfinite(boot)]
    if len(boot) < 10:
        return (float(np.mean(boot)) if len(boot) else np.nan, np.nan, np.nan)
    return (float(np.mean(boot)), float(np.percentile(boot, 2.5)), float(np.percentile(boot, 97.5)))


# ---------- Information / Entropy: MI via discretization ----------

def _discretize_series(x, n_bins=10):
    x = pd.Series(x).astype(float)
    x = x.replace([np.inf, -np.inf], np.nan)
    x = x.dropna()
    if x.empty:
        return None
    # Use quantile bins with fallbacks
    try:
        bins = pd.qcut(x, q=min(n_bins, x.nunique()), duplicates='drop')
        return bins
    except Exception:
        # fallback to uniform bins
        bins = pd.cut(x, bins=min(n_bins, x.nunique()))
        return bins


def mutual_information_discrete(x_disc, y_disc):
    # x_disc and y_disc are pandas Categorical-like (or any hashable)
    x = pd.Series(x_disc)
    y = pd.Series(y_disc)
    df = pd.DataFrame({'x': x, 'y': y}).dropna()
    if df.empty:
        return np.nan, 0
    ct = pd.crosstab(df['x'], df['y'])
    n = ct.to_numpy().sum()
    if n == 0:
        return np.nan, 0
    pxy = ct.to_numpy(dtype=float) / n
    px = pxy.sum(axis=1, keepdims=True)
    py = pxy.sum(axis=0, keepdims=True)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(pxy > 0, pxy / (px @ py), 1.0)
        mi = np.where(pxy > 0, pxy * np.log(ratio), 0.0).sum()
    return float(mi), int(n)


def mi_feature_vs_binary_label(x, y, is_discrete=False, n_bins=10):
    # y should be 0/1
    y = pd.Series(y).astype(float)
    y = y.replace([np.inf, -np.inf], np.nan)
    mask = np.isfinite(y)
    y = y[mask]
    x = pd.Series(x).iloc[mask.to_numpy()]

    # drop missing
    df = pd.DataFrame({'x': x, 'y': y}).dropna()
    if df.shape[0] < 3:
        return np.nan, int(df.shape[0])

    if is_discrete:
        x_disc = df['x'].astype(str)
    else:
        bins = _discretize_series(df['x'], n_bins=n_bins)
        if bins is None:
            return np.nan, int(df.shape[0])
        x_disc = bins

    y_disc = df['y'].astype(int)
    return mutual_information_discrete(x_disc, y_disc)


def mi_feature_vs_continuous_target(x, y, is_discrete_x=False, n_bins=10):
    y = pd.Series(y).astype(float)
    y = y.replace([np.inf, -np.inf], np.nan)
    mask = np.isfinite(y)
    y = y[mask]
    x = pd.Series(x).iloc[mask.to_numpy()]

    df = pd.DataFrame({'x': x, 'y': y}).dropna()
    if df.shape[0] < 3:
        return np.nan, int(df.shape[0])

    if is_discrete_x:
        x_disc = df['x'].astype(str)
    else:
        x_disc = _discretize_series(df['x'], n_bins=n_bins)
        if x_disc is None:
            return np.nan, int(df.shape[0])

    y_disc = _discretize_series(df['y'], n_bins=n_bins)
    if y_disc is None:
        return np.nan, int(df.shape[0])

    return mutual_information_discrete(x_disc, y_disc)


# ---------- Feature selection based on MI ----------

def rank_features_by_mi(df, features, y, mi_kind, feature_groups=None, n_bins=10):
    # mi_kind: 'mi_visit' (binary y), 'mi_reg' (continuous y)
    rows = []
    for f in features:
        is_discrete = (f == 'sex')
        x = df[f] if f in df.columns else pd.Series([np.nan] * len(df))
        if mi_kind == 'mi_visit':
            mi, n = mi_feature_vs_binary_label(x, y, is_discrete=is_discrete, n_bins=n_bins)
        else:
            mi, n = mi_feature_vs_continuous_target(x, y, is_discrete_x=is_discrete, n_bins=n_bins)
        rows.append({'feature': f, 'mi': mi, 'n': n})

    out = pd.DataFrame(rows)
    out['mi'] = out['mi'].fillna(-np.inf)
    out = out.sort_values('mi', ascending=False)
    if feature_groups is not None:
        out['group'] = out['feature'].map(feature_groups)
    return out


def select_topk_global(mi_rank_df, k):
    k = int(k)
    if k <= 0:
        return []
    df = mi_rank_df.copy()
    df = df[df['mi'] > -np.inf]
    return df['feature'].head(min(k, len(df))).tolist()


def select_topk_by_group(mi_rank_df, group_to_features, k):
    selected = []
    for g, feats in group_to_features.items():
        df = mi_rank_df[mi_rank_df['feature'].isin(feats)].copy()
        df = df[df['mi'] > -np.inf]
        selected.extend(df['feature'].head(min(int(k), len(df))).tolist())
    # dedupe while keeping order
    seen = set()
    out = []
    for f in selected:
        if f not in seen:
            seen.add(f)
            out.append(f)
    return out


# ---------- Models (numpy) ----------

def fit_lda_direction(X, y, shrink='auto'):
    # Two-class LDA direction: w = inv(S + lam I) (mu1 - mu0)
    X = _as_float_array(X)
    y = np.asarray(y).astype(int)
    X0 = X[y == 0]
    X1 = X[y == 1]
    if len(X0) < 2 or len(X1) < 2:
        return None
    mu0 = np.mean(X0, axis=0)
    mu1 = np.mean(X1, axis=0)
    S0 = np.cov(X0, rowvar=False, ddof=1)
    S1 = np.cov(X1, rowvar=False, ddof=1)
    S0 = np.atleast_2d(S0)
    S1 = np.atleast_2d(S1)
    S = 0.5 * (S0 + S1)
    S = np.atleast_2d(S)
    p = int(S.shape[0])
    if shrink == 'auto':
        lam = 1e-3 * float(np.trace(S) / max(p, 1))
    else:
        lam = float(shrink)
    S_reg = S + lam * np.eye(p)
    try:
        w = np.linalg.solve(S_reg, (mu1 - mu0))
    except np.linalg.LinAlgError:
        w = np.linalg.pinv(S_reg) @ (mu1 - mu0)
    return w


def predict_lda_scores(X, w):
    X = _as_float_array(X)
    return X @ _as_float_array(w)


def fit_ridge(X, y, alpha):
    X = _as_float_array(X)
    y = _as_float_array(y)
    n, p = X.shape
    A = X.T @ X + float(alpha) * np.eye(p)
    b = X.T @ y
    try:
        w = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        w = np.linalg.pinv(A) @ b
    return w


def soft_threshold(z, g):
    if z > g:
        return z - g
    if z < -g:
        return z + g
    return 0.0


def fit_elasticnet_cd(X, y, alpha, l1_ratio, max_iter=500, tol=1e-6):
    # Coordinate descent on standardized X and centered y.
    X = _as_float_array(X)
    y = _as_float_array(y)
    n, p = X.shape

    # Center y
    y_mean = np.mean(y)
    y_c = y - y_mean

    w = np.zeros(p, dtype=float)
    X_col_norm = (X ** 2).sum(axis=0) / n

    l1 = float(alpha) * float(l1_ratio)
    l2 = float(alpha) * (1.0 - float(l1_ratio))

    # Precompute X^T y
    XTy = (X.T @ y_c) / n

    for _ in range(int(max_iter)):
        w_old = w.copy()
        # Update each coordinate
        for j in range(p):
            # partial residual correlation
            # rho = (X_j^T (y - Xw + w_j X_j))/n
            # compute using precomputed XTy and X^T X w
            # We'll do a safe residual computation given small n.
            r = y_c - X @ w + X[:, j] * w[j]
            rho = float((X[:, j] @ r) / n)

            denom = X_col_norm[j] + l2
            w[j] = soft_threshold(rho, l1) / (denom if denom != 0 else 1.0)

        if np.max(np.abs(w - w_old)) < tol:
            break

    return w, float(y_mean)


def predict_linear(X, w, intercept=0.0):
    return _as_float_array(X) @ _as_float_array(w) + float(intercept)


def fit_pls1_nipals(X, y, n_components=2, tol=1e-8, max_iter=500):
    # Basic PLS1 (single y) using NIPALS; returns regression coef in original X space.
    X = _as_float_array(X)
    y = _as_float_array(y).reshape(-1, 1)

    # Center
    X_mean = np.mean(X, axis=0, keepdims=True)
    y_mean = np.mean(y, axis=0, keepdims=True)
    Xc = X - X_mean
    yc = y - y_mean

    n, p = Xc.shape
    W = []
    P = []
    Q = []

    X_res = Xc.copy()
    y_res = yc.copy()

    for _ in range(int(n_components)):
        # weights
        w = (X_res.T @ y_res).ravel()
        if np.allclose(w, 0):
            break
        w = w / (np.linalg.norm(w) + 1e-12)
        t = X_res @ w.reshape(-1, 1)
        # loadings
        p_vec = (X_res.T @ t).ravel() / (float(t.T @ t) + 1e-12)
        q = float((y_res.T @ t) / (float(t.T @ t) + 1e-12))

        X_res = X_res - t @ p_vec.reshape(1, -1)
        y_res = y_res - t * q

        W.append(w)
        P.append(p_vec)
        Q.append(q)

    if not W:
        coef = np.zeros(p)
    else:
        Wm = np.vstack(W).T  # p x k
        Pm = np.vstack(P).T  # p x k
        Qm = np.asarray(Q).reshape(-1, 1)  # k x 1
        # Regression coefficients: B = W (P^T W)^{-1} Q
        try:
            coef = Wm @ np.linalg.inv(Pm.T @ Wm) @ Qm
        except np.linalg.LinAlgError:
            coef = Wm @ np.linalg.pinv(Pm.T @ Wm) @ Qm
        coef = coef.ravel()

    intercept = float(y_mean.ravel() - X_mean.ravel() @ coef)
    return coef, intercept


# ---------- Metrics ----------

def rmse(y_true, y_pred):
    y_true = _as_float_array(y_true)
    y_pred = _as_float_array(y_pred)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() == 0:
        return np.nan
    return float(np.sqrt(np.mean((y_true[mask] - y_pred[mask]) ** 2)))


def r2(y_true, y_pred):
    y_true = _as_float_array(y_true)
    y_pred = _as_float_array(y_pred)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    if mask.sum() < 2:
        return np.nan
    yt = y_true[mask]
    yp = y_pred[mask]
    ss_res = float(np.sum((yt - yp) ** 2))
    ss_tot = float(np.sum((yt - np.mean(yt)) ** 2))
    return np.nan if ss_tot == 0 else float(1.0 - ss_res / ss_tot)


# ---------- Evaluation wrappers ----------

def lda_loocv(df_long_in, feature_cols, subject_col, visit_col='visit', selection_fn=None, shrink='auto'):
    # Returns oof_df with columns: subject, visit, score
    feats_present = [f for f in feature_cols if f in df_long_in.columns]
    if len(feats_present) == 0:
        return {
            'oof_df': pd.DataFrame(),
            'n_subjects': 0,
            'd_score': np.nan,
            'srm': np.nan,
            'd_ci_low': np.nan,
            'd_ci_high': np.nan,
        }
    sub = df_long_in[[subject_col, visit_col] + feats_present].dropna().copy()
    # require paired visits per subject
    counts = sub.groupby(subject_col)[visit_col].nunique()
    valid_subjects = counts[counts == 2].index
    sub = sub[sub[subject_col].isin(valid_subjects)].copy()

    subjects = sub[subject_col].unique()
    oof_rows = []

    for sid in subjects:
        train_df = sub[sub[subject_col] != sid]
        test_df = sub[sub[subject_col] == sid]

        feats = list(feature_cols)
        if selection_fn is not None:
            feats = selection_fn(train_df)
        feats = [f for f in feats if f in train_df.columns]
        if len(feats) == 0:
            continue

        X_train = train_df[feats].values
        y_train = (train_df[visit_col].values == 2).astype(int)
        X_test = test_df[feats].values

        X_train_s, X_test_s, _, _ = standardize_train_test(X_train, X_test)
        w = fit_lda_direction(X_train_s, y_train, shrink=shrink)
        if w is None:
            continue
        scores = predict_lda_scores(X_test_s, w)

        for v, sc in zip(test_df[visit_col].values, scores):
            oof_rows.append({subject_col: sid, visit_col: int(v), 'score': float(sc)})

    oof_df = pd.DataFrame(oof_rows)
    deltas = paired_deltas_from_long(oof_df.rename(columns={'score': 'value'}), subject_col, visit_col, 'value')
    d_out = compute_cohens_d(deltas)
    srm_out = compute_srm(deltas)
    d_mean, d_lo, d_hi = bootstrap_ci_d(oof_df.rename(columns={'score': 'value'}), subject_col, visit_col, 'value')

    return {
        'oof_df': oof_df,
        'n_subjects': int(d_out['n']),
        'd_score': d_out['d'],
        'srm': srm_out['srm'],
        'd_ci_low': d_lo,
        'd_ci_high': d_hi,
    }


def tune_and_run_regression_loocv(df, feature_cols, target_col, subject_col, model_kind, selection_fn=None,
                                 inner_folds=3, alphas=None, l1_ratios=None, n_components_list=None):
    # df: 1 row per subject; returns oof preds and metrics.
    feats_present = [f for f in feature_cols if f in df.columns]
    if len(feats_present) == 0:
        return {
            'oof_df': pd.DataFrame(),
            'chosen_params_df': pd.DataFrame(),
            'n_subjects': 0,
            'rmse': np.nan,
            'r2': np.nan,
        }
    sub = df[[subject_col, target_col] + feats_present].dropna().copy()
    subjects = sub[subject_col].unique()

    oof_rows = []
    chosen_rows = []

    if alphas is None:
        # Small grid for practicality (secondary metric)
        alphas = np.array([0.01, 0.1, 1.0, 10.0])
    if l1_ratios is None:
        l1_ratios = [0.1, 0.5, 0.9]
    if n_components_list is None:
        n_components_list = [1, 2, 3]

    for sid in subjects:
        train_df = sub[sub[subject_col] != sid]
        test_df = sub[sub[subject_col] == sid]

        feats = list(feature_cols)
        if selection_fn is not None:
            feats = selection_fn(train_df)
        feats = [f for f in feats if f in train_df.columns]
        if len(feats) == 0:
            continue

        X_train = train_df[feats].values
        y_train = train_df[target_col].values
        X_test = test_df[feats].values
        y_test = test_df[target_col].values

        # Standardize X in outer fold
        X_train_s, X_test_s, _, _ = standardize_train_test(X_train, X_test)

        # Inner CV tuning by RMSE on training subjects
        best = None
        best_params = None

        groups_train = train_df[subject_col].values
        for tr_idx, va_idx in group_kfold_indices(groups_train, n_splits=inner_folds, seed=RANDOM_SEED):
            pass  # just to validate generator

        # We re-generate splits per candidate to keep code simple (small n)
        def score_candidate(params):
            rmses = []
            for tr_idx, va_idx in group_kfold_indices(groups_train, n_splits=inner_folds, seed=RANDOM_SEED):
                Xt, Xv = X_train_s[tr_idx], X_train_s[va_idx]
                yt, yv = y_train[tr_idx], y_train[va_idx]

                if model_kind == 'ridge':
                    w = fit_ridge(Xt, yt, alpha=params['alpha'])
                    pred = predict_linear(Xv, w)
                elif model_kind == 'elasticnet':
                    w, b0 = fit_elasticnet_cd(Xt, yt, alpha=params['alpha'], l1_ratio=params['l1_ratio'])
                    pred = predict_linear(Xv, w, intercept=b0)
                elif model_kind == 'pls':
                    coef, intercept = fit_pls1_nipals(Xt, yt, n_components=params['n_components'])
                    pred = predict_linear(Xv, coef, intercept=intercept)
                else:
                    raise ValueError('unknown model_kind')

                rmses.append(rmse(yv, pred))

            rmses = [r for r in rmses if np.isfinite(r)]
            return np.inf if not rmses else float(np.mean(rmses))

        candidates = []
        if model_kind == 'ridge':
            candidates = [{'alpha': float(a)} for a in alphas]
        elif model_kind == 'elasticnet':
            candidates = [{'alpha': float(a), 'l1_ratio': float(l)} for a in alphas for l in l1_ratios]
        elif model_kind == 'pls':
            max_k = min(int(max(n_components_list)), len(feats), max(1, len(train_df) - 1))
            cand = [k for k in n_components_list if 1 <= k <= max_k]
            candidates = [{'n_components': int(k)} for k in cand]
        else:
            raise ValueError('unknown model_kind')

        for params in candidates:
            s = score_candidate(params)
            if best is None or s < best:
                best = s
                best_params = params

        # Fit final model on outer train with best params
        if best_params is None:
            continue

        if model_kind == 'ridge':
            w = fit_ridge(X_train_s, y_train, alpha=best_params['alpha'])
            pred = predict_linear(X_test_s, w)
        elif model_kind == 'elasticnet':
            w, b0 = fit_elasticnet_cd(X_train_s, y_train, alpha=best_params['alpha'], l1_ratio=best_params['l1_ratio'])
            pred = predict_linear(X_test_s, w, intercept=b0)
        else:  # pls
            coef, intercept = fit_pls1_nipals(X_train_s, y_train, n_components=best_params['n_components'])
            pred = predict_linear(X_test_s, coef, intercept=intercept)

        oof_rows.append({subject_col: sid, 'y_true': float(y_test[0]), 'y_pred': float(pred[0])})
        chosen_rows.append({'test_subject': sid, **best_params})

    oof_df = pd.DataFrame(oof_rows)
    if oof_df.empty:
        return {
            'oof_df': oof_df,
            'n_subjects': 0,
            'd_score': np.nan,
            'srm': np.nan,
            'd_ci_low': np.nan,
            'd_ci_high': np.nan,
        }
    chosen_df = pd.DataFrame(chosen_rows)

    return {
        'oof_df': oof_df,
        'chosen_params_df': chosen_df,
        'n_subjects': int(len(oof_df)),
        'rmse': rmse(oof_df['y_true'], oof_df['y_pred']) if len(oof_df) else np.nan,
        'r2': r2(oof_df['y_true'], oof_df['y_pred']) if len(oof_df) else np.nan,
    }


## NEW SECTION — Individual-feature entropy / mutual information analysis

We compute individual-feature MI scores for:
- `mi_visit`: MI(feature, visit)
- `mi_fars1`: MI(feature@V1, FARS1)
- `mi_dfars`: MI(feature_change, ΔFARS)

These are used later for entropy-guided feature selection.


In [5]:
# Build individual-feature MI table

# Feature membership mapping (allow structural features to appear in both structural and structural_ext)
feature_memberships = {}
for g, feats in FEATURE_GROUPS.items():
    for f in feats:
        feature_memberships.setdefault(f, set()).add(g)

# For MI computations, we use the global pool
features_for_mi = ALL_FEATURES

# Compute mi_visit on df_long (binary visit label)
mi_visit_rows = []
visit_y = (df_long['visit'].values == 2).astype(int)
for f in features_for_mi:
    if f not in df_long.columns:
        mi, n = (np.nan, 0)
    else:
        mi, n = mi_feature_vs_binary_label(df_long[f], visit_y, is_discrete=(f == 'sex'))
    mi_visit_rows.append({'feature': f, 'mi_visit': mi, 'n_visit': n})
mi_visit_df = pd.DataFrame(mi_visit_rows)

# Compute mi_fars1 on df_baseline
mi_fars_rows = []
for f in features_for_mi:
    if f not in df_baseline.columns:
        mi, n = (np.nan, 0)
    else:
        mi, n = mi_feature_vs_continuous_target(df_baseline[f], df_baseline['FARS1'], is_discrete_x=(f == 'sex'))
    mi_fars_rows.append({'feature': f, 'mi_fars1': mi, 'n_fars1': n})
mi_fars_df = pd.DataFrame(mi_fars_rows)

# Compute mi_dfars on df_delta
mi_dfars_rows = []
for f in features_for_mi:
    if f not in df_delta.columns:
        mi, n = (np.nan, 0)
    else:
        mi, n = mi_feature_vs_continuous_target(df_delta[f], df_delta['dFARS'], is_discrete_x=(f == 'sex'))
    mi_dfars_rows.append({'feature': f, 'mi_dfars': mi, 'n_dfars': n})
mi_dfars_df = pd.DataFrame(mi_dfars_rows)

# Merge
mi_df = mi_visit_df.merge(mi_fars_df, on='feature', how='outer').merge(mi_dfars_df, on='feature', how='outer')

# Expand to one row per (feature, group) membership (structural features appear in both structural and structural_ext)
rows = []
for _, r in mi_df.iterrows():
    f = r['feature']
    groups = sorted(feature_memberships.get(f, []))
    if not groups:
        continue
    for g in groups:
        rows.append({
            'feature': f,
            'group': g,
            'mi_visit': r.get('mi_visit'),
            'n_visit': r.get('n_visit'),
            'mi_fars1': r.get('mi_fars1'),
            'n_fars1': r.get('n_fars1'),
            'mi_dfars': r.get('mi_dfars'),
            'n_dfars': r.get('n_dfars'),
        })

single_feature_entropy_df = pd.DataFrame(rows)

# Save
entropy_path = OUTPUT_DIR / 'single_feature_entropy.csv'
single_feature_entropy_df.to_csv(entropy_path, index=False)
print('Saved:', entropy_path)

# Display ranked tables
print('\nTop 20 by mi_visit')
display(single_feature_entropy_df.sort_values('mi_visit', ascending=False).head(20))

print('\nTop 20 by mi_fars1')
display(single_feature_entropy_df.sort_values('mi_fars1', ascending=False).head(20))

print('\nTop 20 by mi_dfars')
display(single_feature_entropy_df.sort_values('mi_dfars', ascending=False).head(20))


Saved: results/single_feature_entropy.csv

Top 20 by mi_visit


,feature,group,mi_visit,n_visit,mi_fars1,n_fars1,mi_dfars,n_dfars
36,SupPostCBLM,structural,0.144914,52,1.417027,26,1.327852,26
37,SupPostCBLM,structural_ext,0.144914,52,1.417027,26,1.327852,26
11,FASCP,diffusion,0.143062,52,1.470346,26,1.221213,26
42,dur,background,0.137858,52,1.417027,26,1.054201,26
17,ICP,structural_ext,0.136527,52,1.417027,26,1.221213,26
16,ICP,structural,0.136527,52,1.417027,26,1.221213,26
40,age,background,0.131324,52,1.363708,26,0.934493,26
31,RDICP,diffusion,0.119930,52,1.470346,26,1.401295,26
3,AntCBLM,structural,0.105009,52,1.363708,26,1.454614,26
4,AntCBLM,structural_ext,0.105009,52,1.363708,26,1.454614,26



Top 20 by mi_fars1


,feature,group,mi_visit,n_visit,mi_fars1,n_fars1,mi_dfars,n_dfars
6,CSA_C2,structural_ext,0.061752,52,1.543790,26,1.401295,26
0,ADICP,diffusion,0.071815,52,1.470346,26,1.541128,26
11,FASCP,diffusion,0.143062,52,1.470346,26,1.221213,26
12,FlocCBLM,structural,0.055218,52,1.470346,26,1.381171,26
13,FlocCBLM,structural_ext,0.055218,52,1.470346,26,1.381171,26
35,SCP,structural_ext,0.078349,52,1.470346,26,1.327852,26
5,CSA_C1,structural_ext,0.032086,52,1.470346,26,1.274533,26
34,SCP,structural,0.078349,52,1.470346,26,1.327852,26
31,RDICP,diffusion,0.119930,52,1.470346,26,1.401295,26
9,FAICP,diffusion,0.071815,52,1.470346,26,1.327852,26



Top 20 by mi_dfars


,feature,group,mi_visit,n_visit,mi_fars1,n_fars1,mi_dfars,n_dfars
0,ADICP,diffusion,0.071815,52,1.470346,26,1.541128,26
3,AntCBLM,structural,0.105009,52,1.363708,26,1.454614,26
4,AntCBLM,structural_ext,0.105009,52,1.363708,26,1.454614,26
24,MDSCP,diffusion,0.093271,52,1.310389,26,1.454614,26
26,Medulla,structural_ext,0.028558,52,1.363708,26,1.434490,26
25,Medulla,structural,0.028558,52,1.363708,26,1.434490,26
31,RDICP,diffusion,0.119930,52,1.470346,26,1.401295,26
6,CSA_C2,structural_ext,0.061752,52,1.543790,26,1.401295,26
12,FlocCBLM,structural,0.055218,52,1.470346,26,1.381171,26
13,FlocCBLM,structural_ext,0.055218,52,1.470346,26,1.381171,26


## Single-feature benchmark — Cohen’s d and SRM on raw features

Compute paired V2−V1 deltas for each feature and rank by d/SRM.


In [6]:
# Single-feature d/SRM benchmark on raw feature values

single_rows = []

# Ensure paired subjects only
paired = df_long[df_long[subject_col].isin(paired_subjects)].copy()

for f in ALL_FEATURES:
    if f not in paired.columns:
        continue
    tmp = paired[[subject_col, 'visit', f]].dropna().copy()
    if tmp.empty:
        continue

    deltas = paired_deltas_from_long(tmp.rename(columns={f: 'value'}), subject_col, 'visit', 'value')
    d_out = compute_cohens_d(deltas)
    srm_out = compute_srm(deltas)

    # bootstrap CI for d on deltas directly
    # Build oof-like df for reuse
    tmp_oof = tmp.rename(columns={f: 'value'})
    _, d_lo, d_hi = bootstrap_ci_d(tmp_oof, subject_col, 'visit', 'value')

    # Assign a primary group for summary (background > structural > structural_ext > diffusion)
    # This does not affect membership in entropy table.
    if f in background:
        g = 'background'
    elif f in structural:
        g = 'structural'
    elif f in structural_ext:
        g = 'structural_ext'
    elif f in diffusion:
        g = 'diffusion'
    else:
        g = 'unknown'

    single_rows.append({
        'feature': f,
        'group': g,
        'd_feature': d_out['d'],
        'srm_feature': srm_out['srm'],
        'd_ci_low': d_lo,
        'd_ci_high': d_hi,
        'n_subjects': d_out['n'],
        'mean_diff': d_out['mean'],
        'sd_diff': d_out['sd'],
    })

single_feature_d_df = pd.DataFrame(single_rows).sort_values('d_feature', ascending=False)

single_d_path = OUTPUT_DIR / 'single_feature_d.csv'
single_feature_d_df.to_csv(single_d_path, index=False)
print('Saved:', single_d_path)

display(single_feature_d_df.head(30))


Saved: results/single_feature_d.csv


,feature,group,d_feature,srm_feature,d_ci_low,d_ci_high,n_subjects,mean_diff,sd_diff
29,age,background,16.665773,16.665773,12.621667,27.303488,26,1.952564,0.117160
31,dur,background,7.030795,7.030795,4.818541,14.234868,26,1.985513,0.282402
8,FAICP,diffusion,0.229760,0.229760,-0.138736,0.700958,26,0.005020,0.021847
23,RDICP,diffusion,0.171096,0.171096,-0.254612,0.521056,26,0.000019,0.000108
6,ECC_C1,structural_ext,0.157248,0.157248,-0.242325,0.577288,26,0.004211,0.026779
10,FASCP,diffusion,0.129891,0.129891,-0.289977,0.552729,26,0.002573,0.019810
17,MDICP,diffusion,0.122145,0.122145,-0.326127,0.481423,26,0.000016,0.000130
0,ADICP,diffusion,0.117491,0.117491,-0.295941,0.524043,26,0.000015,0.000131
9,FAMCP,diffusion,0.054658,0.054658,-0.348037,0.462663,26,0.001020,0.018666
28,VermisCBLM,structural,-0.122118,-0.122118,-0.503599,0.295843,26,-30.846923,252.600205


## Entropy-guided feature selection registry (descriptive)

We define top‑k selections using the **individual-feature MI** table, for transparency.

**Important:** leakage-safe model runs recompute MI *inside each outer CV fold on training data only*.
The registry here is a descriptive “global MI” reference.


In [7]:
# Build descriptive selected-features registry from global MI table

def _global_rank(source_col, group=None):
    df = single_feature_entropy_df.copy()
    if group is not None:
        df = df[df['group'] == group]
    df = df.dropna(subset=[source_col]).copy()
    df = df.sort_values(source_col, ascending=False)
    # dedupe features (since some appear in both structural and structural_ext rows)
    seen = set()
    feats = []
    for f in df['feature']:
        if f not in seen:
            seen.add(f)
            feats.append(f)
    return feats

registry_rows = []

for task, source in [
    ('lda_visit', 'mi_visit'),
    ('reg_fars1', 'mi_fars1'),
    ('reg_dfars', 'mi_dfars'),
]:
    # global ranking
    global_rank = _global_rank(source)
    for k in K_VALUES:
        k2 = min(int(k), len(global_rank))
        registry_rows.append({
            'task': task,
            'selection_mode': 'entropy_topk_global',
            'entropy_source': source,
            'k_selected': k2,
            'selected_features': str(global_rank[:k2]),
        })

    # group ranking (union of top-k per group)
    group_to_rank = {g: _global_rank(source, group=g) for g in FEATURE_GROUPS.keys()}
    for k in K_VALUES:
        selected = []
        for g, rank in group_to_rank.items():
            selected.extend(rank[:min(int(k), len(rank))])
        # dedupe
        seen = set()
        sel = []
        for f in selected:
            if f not in seen:
                seen.add(f)
                sel.append(f)
        registry_rows.append({
            'task': task,
            'selection_mode': 'entropy_topk_group',
            'entropy_source': source,
            'k_selected': int(k),
            'selected_features': str(sel),
        })

entropy_selected_feature_sets_df = pd.DataFrame(registry_rows)

reg_path = OUTPUT_DIR / 'entropy_selected_feature_sets.csv'
entropy_selected_feature_sets_df.to_csv(reg_path, index=False)
print('Saved:', reg_path)

display(entropy_selected_feature_sets_df.head(20))


Saved: results/entropy_selected_feature_sets.csv


,task,selection_mode,entropy_source,k_selected,selected_features
0,lda_visit,entropy_topk_global,mi_visit,1,['SupPostCBLM']
1,lda_visit,entropy_topk_global,mi_visit,2,"['SupPostCBLM', 'FASCP']"
2,lda_visit,entropy_topk_global,mi_visit,3,"['SupPostCBLM', 'FASCP', 'dur']"
3,lda_visit,entropy_topk_global,mi_visit,5,"['SupPostCBLM', 'FASCP', 'dur', 'ICP', 'age']"
4,lda_visit,entropy_topk_group,mi_visit,1,"['dur', 'SupPostCBLM', 'FASCP']"
5,lda_visit,entropy_topk_group,mi_visit,2,"['dur', 'age', 'SupPostCBLM', 'ICP', 'FASCP', ..."
6,lda_visit,entropy_topk_group,mi_visit,3,"['dur', 'age', 'GAA1', 'SupPostCBLM', 'ICP', '..."
7,lda_visit,entropy_topk_group,mi_visit,5,"['dur', 'age', 'GAA1', 'GAA2', 'age_at_onset',..."
8,reg_fars1,entropy_topk_global,mi_fars1,1,['CSA_C2']
9,reg_fars1,entropy_topk_global,mi_fars1,2,"['CSA_C2', 'ADICP']"


## SECTION D — Core model runs (full + entropy-selected + single-feature reference rows)

Models:
- Separation: **LDA** (visit label used only to learn 1D axis)
- Regression: **ElasticNet**, **Ridge**, **PLS** for **FARS1** and **ΔFARS**

Entropy-selected runs use MI computed **inside each outer CV fold on training data only**.


In [8]:
# Core model runs

results = []

# Helper to build leakage-safe selection functions for a given task and mode
feature_to_groups = {f: sorted(feature_memberships.get(f, [])) for f in ALL_FEATURES}

def make_selection_fn(task, selection_mode, k_selected, entropy_source):
    k_selected = int(k_selected)

    def select_for_train(train_df):
        # train_df is either long_df (for lda) or baseline/delta df (for regression)
        if task == 'lda_visit':
            y = (train_df['visit'].values == 2).astype(int)
            mi_rank = rank_features_by_mi(train_df, ALL_FEATURES, y, mi_kind='mi_visit')
        elif task == 'reg_fars1':
            y = train_df['FARS1'].values
            mi_rank = rank_features_by_mi(train_df, ALL_FEATURES, y, mi_kind='mi_reg')
        else:
            y = train_df['dFARS'].values
            mi_rank = rank_features_by_mi(train_df, ALL_FEATURES, y, mi_kind='mi_reg')

        if selection_mode == 'entropy_topk_global':
            return select_topk_global(mi_rank, k_selected)

        # entropy_topk_group
        group_to_feats = FEATURE_GROUPS
        return select_topk_by_group(mi_rank, group_to_feats, k_selected)

    return select_for_train


# ---- A) LDA separation: full feature sets ----
for fs_name, feats in FEATURE_SETS.items():
    res = lda_loocv(df_long, feats, subject_col=subject_col, visit_col='visit', selection_fn=None)
    best_single_d = float(single_feature_d_df['d_feature'].max()) if len(single_feature_d_df) else np.nan
    results.append({
        'method': 'LDA',
        'task': 'separation',
        'target': 'visit_axis',
        'feature_set': fs_name,
        'n_features': len(feats),
        'selection_mode': 'full',
        'k_selected': np.nan,
        'entropy_source': np.nan,
        'd_score': res['d_score'],
        'srm': res['srm'],
        'd_ci_low': res['d_ci_low'],
        'd_ci_high': res['d_ci_high'],
        'rmse': np.nan,
        'r2': np.nan,
        'n_subjects': res['n_subjects'],
        'beats_best_single': bool(res['d_score'] > best_single_d) if np.isfinite(res['d_score']) and np.isfinite(best_single_d) else np.nan,
        'notes': '',
    })


# ---- A) LDA separation: entropy-selected subsets ----
for selection_mode in ['entropy_topk_group', 'entropy_topk_global']:
    for k in K_VALUES:
        sel_fn = make_selection_fn('lda_visit', selection_mode, k, entropy_source='mi_visit')
        res = lda_loocv(df_long, ALL_FEATURES, subject_col=subject_col, visit_col='visit', selection_fn=sel_fn)
        best_single_d = float(single_feature_d_df['d_feature'].max()) if len(single_feature_d_df) else np.nan
        results.append({
            'method': 'LDA',
            'task': 'separation',
            'target': 'visit_axis',
            'feature_set': 'global_pool',
            'n_features': np.nan,  # fold-dependent
            'selection_mode': selection_mode,
            'k_selected': int(k),
            'entropy_source': 'mi_visit',
            'd_score': res['d_score'],
            'srm': res['srm'],
            'd_ci_low': res['d_ci_low'],
            'd_ci_high': res['d_ci_high'],
            'rmse': np.nan,
            'r2': np.nan,
            'n_subjects': res['n_subjects'],
            'beats_best_single': bool(res['d_score'] > best_single_d) if np.isfinite(res['d_score']) and np.isfinite(best_single_d) else np.nan,
            'notes': 'entropy-selected (fold-specific)',
        })


# ---- A) LDA single-feature reference rows ----
if len(single_feature_d_df):
    best_by_d = single_feature_d_df.sort_values('d_feature', ascending=False).iloc[0]['feature']
    # Use global mi_visit ranking for reference
    best_by_mi_visit = single_feature_entropy_df.sort_values('mi_visit', ascending=False).iloc[0]['feature']

    for feat, note in [(best_by_d, 'single_feature_best_d'), (best_by_mi_visit, 'single_feature_best_mi_visit')]:
        res = lda_loocv(df_long, [feat], subject_col=subject_col, visit_col='visit', selection_fn=None)
        best_single_d = float(single_feature_d_df['d_feature'].max())
        results.append({
            'method': 'LDA',
            'task': 'separation',
            'target': 'visit_axis',
            'feature_set': feat,
            'n_features': 1,
            'selection_mode': 'single_feature',
            'k_selected': 1,
            'entropy_source': np.nan,
            'd_score': res['d_score'],
            'srm': res['srm'],
            'd_ci_low': res['d_ci_low'],
            'd_ci_high': res['d_ci_high'],
            'rmse': np.nan,
            'r2': np.nan,
            'n_subjects': res['n_subjects'],
            'beats_best_single': bool(res['d_score'] >= best_single_d) if np.isfinite(res['d_score']) else np.nan,
            'notes': note,
        })


# ---- Regression tasks: helper to run full + entropy-selected ----

def run_regression_suite(df_task, target_col, task_name, entropy_source):
    # Full feature sets
    for fs_name, feats in FEATURE_SETS.items():
        for model_kind, method in [('elasticnet', 'ElasticNet'), ('ridge', 'Ridge'), ('pls', 'PLS')]:
            res = tune_and_run_regression_loocv(
                df_task,
                feats,
                target_col=target_col,
                subject_col=subject_col,
                model_kind=model_kind,
                selection_fn=None,
            )
            results.append({
                'method': method,
                'task': 'regression',
                'target': target_col,
                'feature_set': fs_name,
                'n_features': len(feats),
                'selection_mode': 'full',
                'k_selected': np.nan,
                'entropy_source': np.nan,
                'd_score': np.nan,
                'srm': np.nan,
                'd_ci_low': np.nan,
                'd_ci_high': np.nan,
                'rmse': res['rmse'],
                'r2': res['r2'],
                'n_subjects': res['n_subjects'],
                'beats_best_single': np.nan,
                'notes': '',
            })

    # Entropy-selected subsets
    for selection_mode in ['entropy_topk_group', 'entropy_topk_global']:
        for k in K_VALUES:
            sel_fn = make_selection_fn(task_name, selection_mode, k, entropy_source=entropy_source)
            for model_kind, method in [('elasticnet', 'ElasticNet'), ('ridge', 'Ridge'), ('pls', 'PLS')]:
                res = tune_and_run_regression_loocv(
                    df_task,
                    ALL_FEATURES,
                    target_col=target_col,
                    subject_col=subject_col,
                    model_kind=model_kind,
                    selection_fn=sel_fn,
                )
                results.append({
                    'method': method,
                    'task': 'regression',
                    'target': target_col,
                    'feature_set': 'global_pool',
                    'n_features': np.nan,
                    'selection_mode': selection_mode,
                    'k_selected': int(k),
                    'entropy_source': entropy_source,
                    'd_score': np.nan,
                    'srm': np.nan,
                    'd_ci_low': np.nan,
                    'd_ci_high': np.nan,
                    'rmse': res['rmse'],
                    'r2': res['r2'],
                    'n_subjects': res['n_subjects'],
                    'beats_best_single': np.nan,
                    'notes': 'entropy-selected (fold-specific)',
                })


# Run regression suites
run_regression_suite(df_baseline, target_col='FARS1', task_name='reg_fars1', entropy_source='mi_fars1')
run_regression_suite(df_delta, target_col='dFARS', task_name='reg_dfars', entropy_source='mi_dfars')


results_df = pd.DataFrame(results)
print('Total result rows:', len(results_df))

# Save full results and entropy-only subset
out_all = OUTPUT_DIR / 'methods_extension_results.csv'
results_df.to_csv(out_all, index=False)
print('Saved:', out_all)

entropy_only = results_df[results_df['selection_mode'].isin(['entropy_topk_group', 'entropy_topk_global'])].copy()
out_entropy = OUTPUT_DIR / 'entropy_selected_results.csv'
entropy_only.to_csv(out_entropy, index=False)
print('Saved:', out_entropy)

# Display key tables
print('\nTop separation (LDA) by d_score:')
display(results_df[results_df['task']=='separation'].sort_values(['d_score'], ascending=False).head(20))

print('\nTop regression (FARS1) by RMSE (secondary):')
display(results_df[(results_df['task']=='regression') & (results_df['target']=='FARS1')].sort_values(['rmse'], ascending=True).head(20))

print('\nTop regression (dFARS) by RMSE (secondary):')
display(results_df[(results_df['task']=='regression') & (results_df['target']=='dFARS')].sort_values(['rmse'], ascending=True).head(20))


/var/folders/2y/d2x11n9s4sbc3j2svdb5qdb00000gn/T/ipykernel_29311/2253402240.py:352: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  p_vec = (X_res.T @ t).ravel() / (float(t.T @ t) + 1e-12)
/var/folders/2y/d2x11n9s4sbc3j2svdb5qdb00000gn/T/ipykernel_29311/2253402240.py:353: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  q = float((y_res.T @ t) / (float(t.T @ t) + 1e-12))
/var/folders/2y/d2x11n9s4sbc3j2svdb5qdb00000gn/T/ipykernel_29311/2253402240.py:375: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operatio

Total result rows: 135
Saved: results/methods_extension_results.csv
Saved: results/entropy_selected_results.csv

Top separation (LDA) by d_score:


/var/folders/2y/d2x11n9s4sbc3j2svdb5qdb00000gn/T/ipykernel_29311/2253402240.py:352: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  p_vec = (X_res.T @ t).ravel() / (float(t.T @ t) + 1e-12)
/var/folders/2y/d2x11n9s4sbc3j2svdb5qdb00000gn/T/ipykernel_29311/2253402240.py:353: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  q = float((y_res.T @ t) / (float(t.T @ t) + 1e-12))
/var/folders/2y/d2x11n9s4sbc3j2svdb5qdb00000gn/T/ipykernel_29311/2253402240.py:375: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operatio

,method,task,target,feature_set,n_features,selection_mode,k_selected,entropy_source,d_score,srm,d_ci_low,d_ci_high,rmse,r2,n_subjects,beats_best_single,notes
19,LDA,separation,visit_axis,age,1.0,single_feature,1.0,NaN,13.647924,13.647924,11.298347,18.536549,NaN,NaN,26,False,single_feature_best_d
20,LDA,separation,visit_axis,SupPostCBLM,1.0,single_feature,1.0,NaN,0.637710,0.637710,0.279401,1.129694,NaN,NaN,26,False,single_feature_best_mi_visit
1,LDA,separation,visit_axis,structural,11.0,full,NaN,NaN,0.549926,0.549926,0.153358,1.139948,NaN,NaN,26,False,
7,LDA,separation,visit_axis,structural_diffusion,23.0,full,NaN,NaN,0.447783,0.447783,0.084640,0.867848,NaN,NaN,26,False,
2,LDA,separation,visit_axis,structural_ext,15.0,full,NaN,NaN,0.415597,0.415597,0.007436,1.206428,NaN,NaN,26,False,
8,LDA,separation,visit_axis,structural_ext_diffusion,27.0,full,NaN,NaN,0.326244,0.326244,-0.042221,0.791185,NaN,NaN,26,False,
3,LDA,separation,visit_axis,diffusion,12.0,full,NaN,NaN,0.087423,0.087423,-0.285072,0.526933,NaN,NaN,26,False,
0,LDA,separation,visit_axis,background,6.0,full,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,
4,LDA,separation,visit_axis,background_structural,17.0,full,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,
5,LDA,separation,visit_axis,background_structural_ext,21.0,full,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,



Top regression (FARS1) by RMSE (secondary):


,method,task,target,feature_set,n_features,selection_mode,k_selected,entropy_source,d_score,srm,d_ci_low,d_ci_high,rmse,r2,n_subjects,beats_best_single,notes
48,ElasticNet,regression,FARS1,background_structural_diffusion,29.0,full,NaN,NaN,NaN,NaN,NaN,NaN,18.239279,0.502320,26,NaN,
41,PLS,regression,FARS1,background_diffusion,18.0,full,NaN,NaN,NaN,NaN,NaN,NaN,18.835422,0.469255,26,NaN,
39,ElasticNet,regression,FARS1,background_diffusion,18.0,full,NaN,NaN,NaN,NaN,NaN,NaN,18.958649,0.462288,26,NaN,
51,ElasticNet,regression,FARS1,background_structural_ext_diffusion,33.0,full,NaN,NaN,NaN,NaN,NaN,NaN,19.281303,0.443830,26,NaN,
42,ElasticNet,regression,FARS1,structural_diffusion,23.0,full,NaN,NaN,NaN,NaN,NaN,NaN,19.634762,0.423252,26,NaN,
21,ElasticNet,regression,FARS1,background,6.0,full,NaN,NaN,NaN,NaN,NaN,NaN,20.480555,0.372493,26,NaN,
63,ElasticNet,regression,FARS1,global_pool,NaN,entropy_topk_group,5.0,mi_fars1,NaN,NaN,NaN,NaN,20.929126,0.344704,26,NaN,entropy-selected (fold-specific)
33,ElasticNet,regression,FARS1,background_structural,17.0,full,NaN,NaN,NaN,NaN,NaN,NaN,21.499300,0.308513,26,NaN,
23,PLS,regression,FARS1,background,6.0,full,NaN,NaN,NaN,NaN,NaN,NaN,21.514601,0.307529,26,NaN,
50,PLS,regression,FARS1,background_structural_diffusion,29.0,full,NaN,NaN,NaN,NaN,NaN,NaN,21.655020,0.298460,26,NaN,



Top regression (dFARS) by RMSE (secondary):


,method,task,target,feature_set,n_features,selection_mode,k_selected,entropy_source,d_score,srm,d_ci_low,d_ci_high,rmse,r2,n_subjects,beats_best_single,notes
123,ElasticNet,regression,dFARS,global_pool,NaN,entropy_topk_global,1.0,mi_dfars,NaN,NaN,NaN,NaN,8.039377,-0.066568,26,NaN,entropy-selected (fold-specific)
126,ElasticNet,regression,dFARS,global_pool,NaN,entropy_topk_global,2.0,mi_dfars,NaN,NaN,NaN,NaN,8.062216,-0.072637,26,NaN,entropy-selected (fold-specific)
108,ElasticNet,regression,dFARS,background_structural_ext_diffusion,33.0,full,NaN,NaN,NaN,NaN,NaN,NaN,8.095831,-0.081600,26,NaN,
93,ElasticNet,regression,dFARS,background_structural_ext,21.0,full,NaN,NaN,NaN,NaN,NaN,NaN,8.095831,-0.081600,26,NaN,
105,ElasticNet,regression,dFARS,background_structural_diffusion,29.0,full,NaN,NaN,NaN,NaN,NaN,NaN,8.095831,-0.081600,26,NaN,
90,ElasticNet,regression,dFARS,background_structural,17.0,full,NaN,NaN,NaN,NaN,NaN,NaN,8.095831,-0.081600,26,NaN,
99,ElasticNet,regression,dFARS,structural_diffusion,23.0,full,NaN,NaN,NaN,NaN,NaN,NaN,8.095831,-0.081600,26,NaN,
102,ElasticNet,regression,dFARS,structural_ext_diffusion,27.0,full,NaN,NaN,NaN,NaN,NaN,NaN,8.095831,-0.081600,26,NaN,
81,ElasticNet,regression,dFARS,structural,11.0,full,NaN,NaN,NaN,NaN,NaN,NaN,8.095831,-0.081600,26,NaN,
84,ElasticNet,regression,dFARS,structural_ext,15.0,full,NaN,NaN,NaN,NaN,NaN,NaN,8.095831,-0.081600,26,NaN,
